In [ ]:
# Public-repository path setup.
# Run from anywhere inside the repository, or set CEFTAZIDIME_PROJECT_ROOT.
import os
from pathlib import Path

def _repo_root():
    env = os.environ.get("CEFTAZIDIME_PROJECT_ROOT")
    if env:
        return Path(env).expanduser().resolve()
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "README.md").exists() and (candidate / "03_Notebooks").exists():
            return candidate
    return here

def _previous_project_root(project_root):
    env = os.environ.get("GENOME_MIC_AMR_PROJECT_ROOT")
    if env:
        return Path(env).expanduser().resolve()
    return (project_root / "external" / "Genome_MIC_AMR_Emergence").resolve()

PROJECT_ROOT = _repo_root()

#@title Cell 20.1 - Overview, paths, and matched-random ablation design
# Purpose:
# Test whether removing each Notebook 19 broad unitig group weakens the
# reconstructed whole-chromosome MIC association more than removing a
# comparable random set of unitigs.
#
# The 12 observed groups differ greatly in size and unitig-frequency content.
# Therefore, for every random benchmark partition, group labels are shuffled
# WITHIN each unitig presence-count stratum (1 to 175 pathogens).
#
# This preserves for every broad group:
# - the exact number of unitigs;
# - the exact distribution of unitig presence counts;
# - the exact kernel denominator contribution sum[p(1-p)].
#
# The only thing randomized is WHICH unitigs occupy those matched group slots.
#
# For each matched random partition:
# 1. reconstruct the 12 random group kernel contributions;
# 2. remove each random group from the full Notebook 18 kernel;
# 3. refit the same REML model to continuous log2 ceftazidime MIC;
# 4. record the fall from the full variance fraction.
#
# Observed Notebook 19 drops are then compared with these matched-random
# distributions.
#
# Primary benchmark: 250 matched random partitions.
# With 12 groups, this gives a minimum empirical p = 1/251 = 0.003984,
# allowing an extreme first-ranked result to remain below BH q = 0.05.
#
# This notebook does not split groups or remove individual unitigs.

from pathlib import Path
import json
import time

import numpy as np
import pandas as pd
from scipy import sparse, optimize
from IPython.display import display
PROJECT_ROOT = _repo_root()
NOTEBOOK_DIR = PROJECT_ROOT / "03_Notebooks" / "04_Genome_Comparison"
RESULTS_TABLE_DIR = PROJECT_ROOT / "05_Results" / "Tables"

UNITIG_DIR = PROJECT_ROOT / "04_Intermediate" / "10_Whole_Chromosome_Unitigs"
UNITIG_MATRIX = UNITIG_DIR / "10_variable_unitig_matrix_176xM.npz"
UNITIG_SAMPLES = UNITIG_DIR / "10_unitig_sample_order.csv"

NB18_DIR = (
    PROJECT_ROOT
    / "04_Intermediate"
    / "18_Collective_Whole_Chromosome_Association"
)

NB18_K = NB18_DIR / "18_whole_sequence_unitig_similarity_matrix.npz"
NB18_SUMMARY = RESULTS_TABLE_DIR / "18_collective_unitig_association_summary.csv"
NB18_QC = RESULTS_TABLE_DIR / "18_collective_unitig_association_final_QC.csv"

NB19_DIR = (
    PROJECT_ROOT
    / "04_Intermediate"
    / "19_Broad_Unitig_Ablation"
)

NB19_ASSIGNMENT = NB19_DIR / "19_unitig_reference_assignment.npz"
NB19_COMPONENTS = NB19_DIR / "19_group_kernel_components.npz"
NB19_GROUP_MANIFEST = RESULTS_TABLE_DIR / "19_broad_ablation_group_manifest.csv"
NB19_FINAL_RESULTS = RESULTS_TABLE_DIR / "19_broad_ablation_final_results.csv"
NB19_QC = RESULTS_TABLE_DIR / "19_broad_ablation_final_QC.csv"

NB20_DIR = (
    PROJECT_ROOT
    / "04_Intermediate"
    / "20_Matched_Random_Ablation_Benchmark"
)

NB20_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

PROGRESS_FILE = (
    NB20_DIR
    / "20_matched_random_ablation_progress.csv.gz"
)

SMOKE_TEST_FILE = (
    RESULTS_TABLE_DIR
    / "20_matched_random_ablation_smoke_test.csv"
)

FINAL_RESULTS = (
    RESULTS_TABLE_DIR
    / "20_matched_random_ablation_final_results.csv"
)

FINAL_QC = (
    RESULTS_TABLE_DIR
    / "20_matched_random_ablation_final_QC.csv"
)

COMPLETION_FILE = (
    NB20_DIR
    / "20_MATCHED_RANDOM_ABLATION_COMPLETE.json"
)

EXPECTED_PATHOGENS = 176
EXPECTED_UNITIGS = 1_287_844
EXPECTED_GROUPS = 12
EXPECTED_FULL_VARIANCE_FRACTION = 0.538815

N_RANDOM_PARTITIONS = 250
RANDOM_SEED_BASE = 2026091400

for path in [
    PROJECT_ROOT,
    NOTEBOOK_DIR,
    RESULTS_TABLE_DIR,
    UNITIG_MATRIX,
    UNITIG_SAMPLES,
    NB18_K,
    NB18_SUMMARY,
    NB18_QC,
    NB19_ASSIGNMENT,
    NB19_COMPONENTS,
    NB19_GROUP_MANIFEST,
    NB19_FINAL_RESULTS,
    NB19_QC,
]:
    assert path.exists(), f"Required input not found: {path}"

print("Notebook 20 - Matched-Random Ablation Benchmark")
print("Pathogens:", EXPECTED_PATHOGENS)
print("Variable unitigs:", f"{EXPECTED_UNITIGS:,}")
print("Broad groups:", EXPECTED_GROUPS)
print("Matched random partitions:", N_RANDOM_PARTITIONS)
print("Matching: exact unitig presence-count distribution within every group")
print("No finer group splitting is performed in this notebook.")
print("\nTransition: Cell 20.2 will verify Notebook 18/19 outputs and load the exact full-kernel decomposition.")


In [ ]:
#@title Cell 20.2 - Verify Notebook 18/19 outputs and recover the exact baseline
# Purpose:
# Confirm the accepted whole-sequence association and broad ablation outputs,
# load the 12 exact kernel contributions, and recover the continuous MIC
# phenotype in the Notebook 10 pathogen order.

nb18_qc = pd.read_csv(
    NB18_QC
)

nb19_qc = pd.read_csv(
    NB19_QC
)

nb18_summary = pd.read_csv(
    NB18_SUMMARY
)

group_manifest = pd.read_csv(
    NB19_GROUP_MANIFEST
)

observed_results = pd.read_csv(
    NB19_FINAL_RESULTS
)

assert len(nb18_qc) == 1
assert len(nb19_qc) == 1
assert len(nb18_summary) == 1

assert bool(
    nb18_qc.loc[
        0,
        "final_QC_pass",
    ]
)

assert bool(
    nb19_qc.loc[
        0,
        "final_QC_pass",
    ]
)

baseline_variance_fraction = float(
    nb18_summary.loc[
        0,
        "whole_sequence_unitig_variance_fraction",
    ]
)

assert abs(
    baseline_variance_fraction
    - EXPECTED_FULL_VARIANCE_FRACTION
) < 0.001

assert len(
    group_manifest
) == EXPECTED_GROUPS

assert len(
    observed_results
) == EXPECTED_GROUPS

samples = pd.read_csv(
    UNITIG_SAMPLES
)

samples = (
    samples
    .sort_values(
        "sample_index"
    )
    .reset_index(
        drop=True
    )
)

assert len(
    samples
) == EXPECTED_PATHOGENS

assert np.array_equal(
    samples[
        "sample_index"
    ].to_numpy(
        dtype=int
    ),
    np.arange(
        EXPECTED_PATHOGENS
    ),
)

y = samples[
    "log2_mic"
].to_numpy(
    dtype=float
)

assert y.shape == (
    EXPECTED_PATHOGENS,
)

assert np.isfinite(
    y
).all()

with np.load(
    NB18_K
) as archive:
    K_full = np.asarray(
        archive[
            "K_unitig"
        ],
        dtype=float,
    )

K_full = (
    K_full
    + K_full.T
) / 2.0

assert K_full.shape == (
    EXPECTED_PATHOGENS,
    EXPECTED_PATHOGENS,
)

with np.load(
    NB19_COMPONENTS
) as archive:
    observed_group_numerators = np.asarray(
        archive[
            "group_numerators"
        ],
        dtype=float,
    )

    observed_group_denominators = np.asarray(
        archive[
            "group_denominators"
        ],
        dtype=float,
    )

    observed_group_unitig_counts = np.asarray(
        archive[
            "group_unitig_counts"
        ],
        dtype=np.int64,
    )

assert observed_group_numerators.shape == (
    EXPECTED_GROUPS,
    EXPECTED_PATHOGENS,
    EXPECTED_PATHOGENS,
)

assert observed_group_denominators.shape == (
    EXPECTED_GROUPS,
)

assert observed_group_unitig_counts.shape == (
    EXPECTED_GROUPS,
)

total_numerator = np.sum(
    observed_group_numerators,
    axis=0,
)

total_denominator = float(
    np.sum(
        observed_group_denominators
    )
)

K_reconstructed = (
    total_numerator
    / total_denominator
)

K_reconstructed = (
    K_reconstructed
    + K_reconstructed.T
) / 2.0

reconstruction_error = float(
    np.max(
        np.abs(
            K_reconstructed
            - K_full
        )
    )
)

assert reconstruction_error < 1e-10

group_manifest = (
    group_manifest
    .sort_values(
        "group_code"
    )
    .reset_index(
        drop=True
    )
)

assert np.array_equal(
    group_manifest[
        "group_code"
    ].to_numpy(
        dtype=int
    ),
    np.arange(
        EXPECTED_GROUPS
    ),
)

assert np.array_equal(
    group_manifest[
        "n_unitigs"
    ].to_numpy(
        dtype=np.int64
    ),
    observed_group_unitig_counts,
)

observed_results = (
    observed_results
    .sort_values(
        "group_code"
    )
    .reset_index(
        drop=True
    )
)

observed_drop = observed_results[
    "absolute_drop_after_removal"
].to_numpy(
    dtype=float
)

print("Notebook 18 QC: PASS")
print("Notebook 19 QC: PASS")
print("Full variance fraction:", baseline_variance_fraction)
print("Full K reconstruction error:", reconstruction_error)
print("Continuous log2 MIC range:", float(y.min()), "to", float(y.max()))

print("\nObserved broad-group drops:")
display(
    observed_results[
        [
            "group_code",
            "group_name",
            "n_unitigs",
            "kernel_denominator_fraction",
            "leave_one_group_out_variance_fraction",
            "absolute_drop_after_removal",
        ]
    ]
)

print("\nCell 20.2 complete.")
print("Transition: Cell 20.3 will verify the exact matching strata used for the random benchmark.")


In [ ]:
#@title Cell 20.3 - Build exact unitig presence-count matching strata
# Purpose:
# Establish the strata used for matched randomization.
#
# Unitigs are stratified by the number of pathogens carrying them.
# Random group labels will be shuffled only among unitigs with the same
# presence count.
#
# Therefore every random counterpart of every observed group has:
# - exactly the same number of unitigs;
# - exactly the same presence-count histogram;
# - exactly the same sum[p(1-p)] kernel denominator contribution.

X = sparse.load_npz(
    UNITIG_MATRIX
).tocsc()

assert X.shape == (
    EXPECTED_PATHOGENS,
    EXPECTED_UNITIGS,
)

with np.load(
    NB19_ASSIGNMENT
) as archive:
    observed_group_code = np.asarray(
        archive[
            "group_code"
        ],
        dtype=np.uint8,
    )

assert observed_group_code.shape == (
    EXPECTED_UNITIGS,
)

assert observed_group_code.min() >= 0
assert observed_group_code.max() < EXPECTED_GROUPS

presence_count = np.asarray(
    X.sum(
        axis=0
    )
).ravel().astype(
    np.int16
)

assert presence_count.shape == (
    EXPECTED_UNITIGS,
)

assert presence_count.min() >= 1
assert presence_count.max() <= (
    EXPECTED_PATHOGENS - 1
)

stratum_indices = {}
stratum_group_counts = {}

for count in range(
    1,
    EXPECTED_PATHOGENS,
):
    indices = np.flatnonzero(
        presence_count
        == count
    )

    if len(
        indices
    ) == 0:
        continue

    counts_by_group = np.bincount(
        observed_group_code[
            indices
        ],
        minlength=EXPECTED_GROUPS,
    ).astype(
        np.int64
    )

    assert int(
        counts_by_group.sum()
    ) == len(
        indices
    )

    stratum_indices[
        count
    ] = indices

    stratum_group_counts[
        count
    ] = counts_by_group

reconstructed_group_counts = np.zeros(
    EXPECTED_GROUPS,
    dtype=np.int64,
)

reconstructed_group_denominators = np.zeros(
    EXPECTED_GROUPS,
    dtype=np.float64,
)

for count, counts_by_group in stratum_group_counts.items():
    p = (
        count
        / EXPECTED_PATHOGENS
    )

    weight = (
        p
        * (
            1.0
            - p
        )
    )

    reconstructed_group_counts += (
        counts_by_group
    )

    reconstructed_group_denominators += (
        counts_by_group
        * weight
    )

assert np.array_equal(
    reconstructed_group_counts,
    observed_group_unitig_counts,
)

assert np.allclose(
    reconstructed_group_denominators,
    observed_group_denominators,
    atol=1e-10,
    rtol=1e-12,
)

matching_summary = pd.DataFrame(
    {
        "group_code": np.arange(
            EXPECTED_GROUPS,
            dtype=int,
        ),
        "group_name": group_manifest[
            "group_name"
        ].tolist(),
        "unitigs": observed_group_unitig_counts,
        "kernel_denominator": observed_group_denominators,
        "matching_unitig_count_exact": (
            reconstructed_group_counts
            == observed_group_unitig_counts
        ),
        "matching_denominator_exact": np.isclose(
            reconstructed_group_denominators,
            observed_group_denominators,
            atol=1e-10,
            rtol=1e-12,
        ),
    }
)

assert matching_summary[
    "matching_unitig_count_exact"
].all()

assert matching_summary[
    "matching_denominator_exact"
].all()

print("Presence-count strata:", len(stratum_indices))
print("Presence-count range:", min(stratum_indices), "to", max(stratum_indices))
print("Exact unitig-count matching: PASS")
print("Exact kernel-denominator matching: PASS")

display(
    matching_summary
)

print("\nCell 20.3 complete.")
print("Transition: Cell 20.4 will run one matched-random smoke test and estimate the full benchmark runtime.")


In [ ]:
#@title Cell 20.4 - One matched-random smoke test and runtime estimate
# Purpose:
# Run one complete matched-random partition before the long benchmark.
#
# This confirms:
# - every random group exactly matches its observed unitig count;
# - every random group exactly matches its observed presence-count histogram;
# - every random group exactly matches its observed kernel denominator;
# - all 12 random leave-one-group-out kernels can be fitted successfully.
#
# The elapsed time is used to estimate the full 250-partition runtime.

def prepare_kernel(K):
    K = np.asarray(
        K,
        dtype=float,
    )

    K = (
        K
        + K.T
    ) / 2.0

    eigenvalues, eigenvectors = np.linalg.eigh(
        K
    )

    minimum_eigenvalue = float(
        eigenvalues.min()
    )

    if minimum_eigenvalue < -1e-6:
        raise ValueError(
            f"Kernel is not positive semidefinite: minimum eigenvalue = {minimum_eigenvalue}"
        )

    eigenvalues = np.maximum(
        eigenvalues,
        0.0,
    )

    transformed_intercept = (
        eigenvectors.T
        @ np.ones(
            K.shape[0],
            dtype=float,
        )
    )

    return {
        "eigenvalues": eigenvalues,
        "eigenvectors": eigenvectors,
        "transformed_intercept": transformed_intercept,
    }

def fit_null_reml_prepared(y_input, prepared):
    y_input = np.asarray(
        y_input,
        dtype=float,
    ).reshape(-1)

    eigenvalues = prepared[
        "eigenvalues"
    ]

    eigenvectors = prepared[
        "eigenvectors"
    ]

    transformed_intercept = prepared[
        "transformed_intercept"
    ]

    transformed_y = (
        eigenvectors.T
        @ y_input
    )

    n = len(
        y_input
    )

    degrees_of_freedom = (
        n - 1
    )

    def evaluate_ratio(ratio):
        if ratio < 0:
            return None

        covariance_eigenvalues = (
            1.0
            + ratio
            * eigenvalues
        )

        if np.any(
            covariance_eigenvalues <= 0
        ):
            return None

        inverse_weights = (
            1.0
            / covariance_eigenvalues
        )

        information = float(
            np.sum(
                transformed_intercept
                * transformed_intercept
                * inverse_weights
            )
        )

        if information <= 0:
            return None

        beta_0 = float(
            np.sum(
                transformed_intercept
                * transformed_y
                * inverse_weights
            )
            / information
        )

        transformed_residual = (
            transformed_y
            - beta_0
            * transformed_intercept
        )

        residual_quadratic = float(
            np.sum(
                transformed_residual
                * transformed_residual
                * inverse_weights
            )
        )

        if residual_quadratic <= 0:
            return None

        sigma_e2 = (
            residual_quadratic
            / degrees_of_freedom
        )

        sigma_g2 = (
            ratio
            * sigma_e2
        )

        objective = 0.5 * (
            degrees_of_freedom
            * np.log(
                sigma_e2
            )
            + np.log(
                covariance_eigenvalues
            ).sum()
            + np.log(
                information
            )
        )

        return {
            "objective": float(
                objective
            ),
            "variance_fraction": float(
                sigma_g2
                / (
                    sigma_g2
                    + sigma_e2
                )
            ),
        }

    def objective_on_log_ratio(log_ratio):
        result = evaluate_ratio(
            np.exp(
                log_ratio
            )
        )

        if result is None:
            return np.inf

        return result[
            "objective"
        ]

    optimized = optimize.minimize_scalar(
        objective_on_log_ratio,
        bounds=(
            -12.0,
            12.0,
        ),
        method="bounded",
        options={
            "xatol": 1e-8,
            "maxiter": 500,
        },
    )

    candidates = []

    zero_result = evaluate_ratio(
        0.0
    )

    if zero_result is not None:
        candidates.append(
            zero_result
        )

    if optimized.success:
        optimized_result = evaluate_ratio(
            float(
                np.exp(
                    optimized.x
                )
            )
        )

        if optimized_result is not None:
            candidates.append(
                optimized_result
            )

    high_result = evaluate_ratio(
        float(
            np.exp(
                12.0
            )
        )
    )

    if high_result is not None:
        candidates.append(
            high_result
        )

    assert candidates

    return min(
        candidates,
        key=lambda item: item[
            "objective"
        ],
    )

def matched_random_group_code(replicate_index):
    rng = np.random.default_rng(
        RANDOM_SEED_BASE
        + int(
            replicate_index
        )
    )

    random_group_code = np.empty(
        EXPECTED_UNITIGS,
        dtype=np.uint8,
    )

    for count, indices in stratum_indices.items():
        counts_by_group = stratum_group_counts[
            count
        ]

        labels = np.repeat(
            np.arange(
                EXPECTED_GROUPS,
                dtype=np.uint8,
            ),
            counts_by_group,
        )

        assert len(
            labels
        ) == len(
            indices
        )

        rng.shuffle(
            labels
        )

        random_group_code[
            indices
        ] = labels

    return random_group_code

def group_numerator_from_columns(columns):
    X_group = X[
        :,
        columns,
    ]

    count_group = presence_count[
        columns
    ].astype(
        np.float64
    )

    p_group = (
        count_group
        / EXPECTED_PATHOGENS
    )

    X_group_int = X_group.astype(
        np.int32
    )

    XX_group = (
        X_group_int
        @ X_group_int.T
    ).toarray().astype(
        np.float64
    )

    Xp_group = np.asarray(
        X_group
        @ p_group
    ).reshape(-1).astype(
        np.float64
    )

    p2_group = float(
        p_group
        @ p_group
    )

    numerator_group = (
        XX_group
        - Xp_group[:, None]
        - Xp_group[None, :]
        + p2_group
    )

    return (
        numerator_group
        + numerator_group.T
    ) / 2.0

smoke_start = time.time()

smoke_group_code = matched_random_group_code(
    0
)

smoke_counts = np.bincount(
    smoke_group_code,
    minlength=EXPECTED_GROUPS,
).astype(
    np.int64
)

assert np.array_equal(
    smoke_counts,
    observed_group_unitig_counts,
)

smoke_rows = []

for group_index in range(
    EXPECTED_GROUPS
):
    columns = np.flatnonzero(
        smoke_group_code
        == group_index
    )

    random_numerator = group_numerator_from_columns(
        columns
    )

    denominator_group = float(
        observed_group_denominators[
            group_index
        ]
    )

    leave_out_K = (
        total_numerator
        - random_numerator
    ) / (
        total_denominator
        - denominator_group
    )

    leave_out_K = (
        leave_out_K
        + leave_out_K.T
    ) / 2.0

    prepared = prepare_kernel(
        leave_out_K
    )

    fit = fit_null_reml_prepared(
        y,
        prepared,
    )

    random_fraction = float(
        fit[
            "variance_fraction"
        ]
    )

    random_drop = (
        baseline_variance_fraction
        - random_fraction
    )

    smoke_rows.append(
        {
            "group_code": group_index,
            "group_name": group_manifest.loc[
                group_index,
                "group_name",
            ],
            "matched_unitigs": len(
                columns
            ),
            "random_leave_one_group_out_variance_fraction": random_fraction,
            "random_drop_from_full": random_drop,
        }
    )

smoke_results = pd.DataFrame(
    smoke_rows
)

smoke_elapsed_seconds = (
    time.time()
    - smoke_start
)

estimated_total_minutes = (
    smoke_elapsed_seconds
    * N_RANDOM_PARTITIONS
    / 60.0
)

smoke_results.to_csv(
    SMOKE_TEST_FILE,
    index=False,
)

print("Matched-random smoke test: PASS")
print("One complete 12-group partition elapsed:", f"{smoke_elapsed_seconds:.1f}", "seconds")
print("Estimated 250-partition runtime:", f"{estimated_total_minutes:.1f}", "minutes")
print("The long benchmark is restartable after every completed partition.")

display(
    smoke_results
)

print("\nCell 20.4 complete.")
print("Transition: Cell 20.5 will run or resume the 250 matched-random partitions.")


In [ ]:
#@title Cell 20.5 - Run or resume 250 matched-random ablation partitions
# Purpose:
# Generate the complete matched-random null distribution for all 12 groups.
#
# Progress is saved to project storage after every completed random partition.
# If Colab disconnects, rerun Cells 20.1-20.5; completed partitions are skipped.

if PROGRESS_FILE.exists():
    progress = pd.read_csv(
        PROGRESS_FILE
    )

else:
    progress = pd.DataFrame(
        columns=[
            "replicate",
            "group_code",
            "group_name",
            "random_leave_one_group_out_variance_fraction",
            "random_drop_from_full",
        ]
    )

completed_replicates = set()

if len(
    progress
) > 0:
    replicate_counts = (
        progress.groupby(
            "replicate"
        )[
            "group_code"
        ]
        .nunique()
    )

    completed_replicates = set(
        replicate_counts.loc[
            replicate_counts
            == EXPECTED_GROUPS
        ].index.astype(
            int
        )
    )

print(
    "Already completed matched-random partitions:",
    len(
        completed_replicates
    ),
    "/",
    N_RANDOM_PARTITIONS,
)

benchmark_start = time.time()

for replicate_index in range(
    N_RANDOM_PARTITIONS
):
    if replicate_index in completed_replicates:
        continue

    replicate_start = time.time()

    random_group_code = matched_random_group_code(
        replicate_index
    )

    random_counts = np.bincount(
        random_group_code,
        minlength=EXPECTED_GROUPS,
    ).astype(
        np.int64
    )

    assert np.array_equal(
        random_counts,
        observed_group_unitig_counts,
    )

    replicate_rows = []

    for group_index in range(
        EXPECTED_GROUPS
    ):
        columns = np.flatnonzero(
            random_group_code
            == group_index
        )

        random_numerator = group_numerator_from_columns(
            columns
        )

        denominator_group = float(
            observed_group_denominators[
                group_index
            ]
        )

        leave_out_K = (
            total_numerator
            - random_numerator
        ) / (
            total_denominator
            - denominator_group
        )

        leave_out_K = (
            leave_out_K
            + leave_out_K.T
        ) / 2.0

        prepared = prepare_kernel(
            leave_out_K
        )

        fit = fit_null_reml_prepared(
            y,
            prepared,
        )

        random_fraction = float(
            fit[
                "variance_fraction"
            ]
        )

        random_drop = (
            baseline_variance_fraction
            - random_fraction
        )

        replicate_rows.append(
            {
                "replicate": replicate_index,
                "group_code": group_index,
                "group_name": group_manifest.loc[
                    group_index,
                    "group_name",
                ],
                "random_leave_one_group_out_variance_fraction": random_fraction,
                "random_drop_from_full": random_drop,
            }
        )

    if len(
        progress
    ) > 0:
        progress = progress.loc[
            progress[
                "replicate"
            ].astype(
                int
            )
            != replicate_index
        ].copy()

    progress = pd.concat(
        [
            progress,
            pd.DataFrame(
                replicate_rows
            ),
        ],
        ignore_index=True,
    )

    progress = (
        progress
        .sort_values(
            [
                "replicate",
                "group_code",
            ]
        )
        .reset_index(
            drop=True
        )
    )

    progress.to_csv(
        PROGRESS_FILE,
        index=False,
        compression="gzip",
    )

    elapsed_replicate = (
        time.time()
        - replicate_start
    )

    print(
        "Completed random partition:",
        replicate_index + 1,
        "/",
        N_RANDOM_PARTITIONS,
        "- elapsed:",
        f"{elapsed_replicate:.1f}",
        "seconds",
    )

progress = pd.read_csv(
    PROGRESS_FILE
)

replicate_counts = (
    progress.groupby(
        "replicate"
    )[
        "group_code"
    ]
    .nunique()
)

assert len(
    replicate_counts
) == N_RANDOM_PARTITIONS

assert (
    replicate_counts
    == EXPECTED_GROUPS
).all()

assert len(
    progress
) == (
    N_RANDOM_PARTITIONS
    * EXPECTED_GROUPS
)

print("\nMatched-random benchmark complete.")
print("Random partitions:", N_RANDOM_PARTITIONS)
print("Saved rows:", len(progress))
print(
    "Current-session elapsed:",
    f"{(time.time() - benchmark_start) / 60.0:.1f}",
    "minutes",
)

print("\nCell 20.5 complete.")
print("Transition: Cell 20.6 will compare each observed ablation drop with its exact matched-random distribution.")


In [ ]:
#@title Cell 20.6 - Compare observed drops with matched-random distributions
# Purpose:
# For every broad group, compare its observed Notebook 19 ablation drop with
# the distribution obtained from matched random unitig groups.
#
# Empirical p:
#   (1 + number of random drops >= observed drop) / (1 + random partitions)
#
# BH correction is applied across the 12 pre-defined broad groups.

progress = pd.read_csv(
    PROGRESS_FILE
)

assert len(
    progress
) == (
    N_RANDOM_PARTITIONS
    * EXPECTED_GROUPS
)

def benjamini_hochberg(p_values):
    p_values = np.asarray(
        p_values,
        dtype=float,
    )

    n = len(
        p_values
    )

    order = np.argsort(
        p_values
    )

    ranked = p_values[
        order
    ]

    adjusted = (
        ranked
        * n
        / np.arange(
            1,
            n + 1,
            dtype=float,
        )
    )

    adjusted = np.minimum.accumulate(
        adjusted[
            ::-1
        ]
    )[
        ::-1
    ]

    adjusted = np.minimum(
        adjusted,
        1.0,
    )

    result = np.empty(
        n,
        dtype=float,
    )

    result[
        order
    ] = adjusted

    return result

result_rows = []

for group_index in range(
    EXPECTED_GROUPS
):
    group_name = str(
        group_manifest.loc[
            group_index,
            "group_name",
        ]
    )

    observed_group_drop = float(
        observed_results.loc[
            observed_results[
                "group_code"
            ]
            == group_index,
            "absolute_drop_after_removal",
        ].iloc[0]
    )

    random_drops = (
        progress.loc[
            progress[
                "group_code"
            ].astype(
                int
            )
            == group_index,
            "random_drop_from_full",
        ]
        .to_numpy(
            dtype=float
        )
    )

    assert len(
        random_drops
    ) == N_RANDOM_PARTITIONS

    n_equal_or_greater = int(
        np.sum(
            random_drops
            >= observed_group_drop
        )
    )

    empirical_p = float(
        (
            1
            + n_equal_or_greater
        )
        / (
            N_RANDOM_PARTITIONS
            + 1
        )
    )

    percentile = float(
        100.0
        * np.mean(
            random_drops
            <= observed_group_drop
        )
    )

    result_rows.append(
        {
            "group_code": group_index,
            "group_name": group_name,
            "n_unitigs": int(
                observed_group_unitig_counts[
                    group_index
                ]
            ),
            "kernel_denominator_fraction": float(
                observed_group_denominators[
                    group_index
                ]
                / total_denominator
            ),
            "observed_drop_after_removal": observed_group_drop,
            "matched_random_mean_drop": float(
                np.mean(
                    random_drops
                )
            ),
            "matched_random_median_drop": float(
                np.median(
                    random_drops
                )
            ),
            "matched_random_95th_percentile": float(
                np.quantile(
                    random_drops,
                    0.95,
                )
            ),
            "matched_random_99th_percentile": float(
                np.quantile(
                    random_drops,
                    0.99,
                )
            ),
            "observed_drop_percentile": percentile,
            "random_drops_equal_or_greater": n_equal_or_greater,
            "empirical_p_value": empirical_p,
        }
    )

final_results = pd.DataFrame(
    result_rows
)

final_results[
    "Benjamini_Hochberg_q_value"
] = benjamini_hochberg(
    final_results[
        "empirical_p_value"
    ].to_numpy(
        dtype=float
    )
)

final_results[
    "larger_drop_than_matched_random_after_BH"
] = (
    final_results[
        "Benjamini_Hochberg_q_value"
    ]
    < 0.05
)

final_results[
    "size_adjusted_priority_rank"
] = (
    final_results[
        "empirical_p_value"
    ]
    .rank(
        method="min",
        ascending=True,
    )
    .astype(
        int
    )
)

final_results = (
    final_results
    .sort_values(
        [
            "empirical_p_value",
            "observed_drop_after_removal",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .reset_index(
        drop=True
    )
)

final_results.to_csv(
    FINAL_RESULTS,
    index=False,
)

display(
    final_results[
        [
            "size_adjusted_priority_rank",
            "group_name",
            "n_unitigs",
            "kernel_denominator_fraction",
            "observed_drop_after_removal",
            "matched_random_median_drop",
            "matched_random_95th_percentile",
            "matched_random_99th_percentile",
            "observed_drop_percentile",
            "empirical_p_value",
            "Benjamini_Hochberg_q_value",
            "larger_drop_than_matched_random_after_BH",
        ]
    ]
)

print("\nCell 20.6 complete.")
print("Transition: Cell 20.7 will perform final QC and state which broad groups, if any, merit finer ablation.")


In [ ]:
#@title Cell 20.7 - Final QC and stopping decision
# Purpose:
# Freeze the matched-random benchmark and decide whether any broad group
# shows an ablation effect larger than expected from a matched amount of
# sequence variation.
#
# Only groups passing BH q < 0.05 are treated as size-adjusted priorities
# for finer ablation.

assert PROGRESS_FILE.exists()
assert SMOKE_TEST_FILE.exists()
assert FINAL_RESULTS.exists()

progress = pd.read_csv(
    PROGRESS_FILE
)

saved_results = pd.read_csv(
    FINAL_RESULTS
)

assert len(
    progress
) == (
    N_RANDOM_PARTITIONS
    * EXPECTED_GROUPS
)

assert len(
    saved_results
) == EXPECTED_GROUPS

assert progress[
    "random_drop_from_full"
].notna().all()

assert saved_results[
    "empirical_p_value"
].between(
    0,
    1,
).all()

assert saved_results[
    "Benjamini_Hochberg_q_value"
].between(
    0,
    1,
).all()

priority_mask = saved_results[
    "larger_drop_than_matched_random_after_BH"
].astype(
    bool
)

priority_groups = saved_results.loc[
    priority_mask,
    "group_name",
].astype(
    str
).tolist()

qc = pd.DataFrame(
    [
        {
            "pathogens": EXPECTED_PATHOGENS,
            "variable_unitigs": EXPECTED_UNITIGS,
            "broad_groups": EXPECTED_GROUPS,
            "random_partitions": N_RANDOM_PARTITIONS,
            "progress_rows": len(
                progress
            ),
            "all_groups_exact_unitig_count_matched": True,
            "all_groups_exact_presence_count_distribution_matched": True,
            "all_groups_exact_kernel_denominator_matched": True,
            "groups_passing_size_adjusted_BH_q_lt_0_05": len(
                priority_groups
            ),
            "final_QC_pass": True,
        }
    ]
)

qc.to_csv(
    FINAL_QC,
    index=False,
)

completion_payload = {
    "status": "complete",
    "pathogens": EXPECTED_PATHOGENS,
    "variable_unitigs": EXPECTED_UNITIGS,
    "broad_groups": EXPECTED_GROUPS,
    "matched_random_partitions": N_RANDOM_PARTITIONS,
    "groups_passing_size_adjusted_BH_q_lt_0_05": priority_groups,
    "final_QC_pass": True,
}

COMPLETION_FILE.write_text(
    json.dumps(
        completion_payload,
        indent=2,
    ),
    encoding="utf-8",
)

print("Final QC: PASS")

print("\nSize-adjusted matched-random ablation results:")
display(
    saved_results[
        [
            "size_adjusted_priority_rank",
            "group_name",
            "observed_drop_after_removal",
            "matched_random_median_drop",
            "matched_random_95th_percentile",
            "observed_drop_percentile",
            "empirical_p_value",
            "Benjamini_Hochberg_q_value",
            "larger_drop_than_matched_random_after_BH",
        ]
    ]
)

if priority_groups:
    print(
        "\nStopping decision: the following broad groups caused a larger "
        "loss of the collective association than matched random sequence "
        "sets after BH correction:"
    )

    for group_name in priority_groups:
        print(
            "-",
            group_name,
        )

    print(
        "\nThese groups are justified priorities for finer ablation. "
        "They are not yet causal regions or causal variant combinations."
    )

else:
    print(
        "\nStopping decision: no broad group caused a larger loss of the "
        "collective association than matched random sequence sets after "
        "BH correction."
    )

    print(
        "This would support a broadly distributed/redundant chromosomal "
        "signal rather than one broad group being disproportionately required."
    )

print(
    "\nNotebook 20 ends here. Review this benchmark before any finer splitting."
)
